# Light EDA / Audit — All Rats

Fast, automated pass over every rat's session, checking the same structural facts we
manually confirmed for Mitt (trial counts, label balance, sampling rate, odor label cleanliness), so we
know before writing preprocessing code whether all five sessions can be handled with one shared pipeline
or whether any rat needs special-case handling.

This file DOES NOT load the `lfp`, `spk`, or `wfm` files. Those are large (LFP alone
is roughly 900MB per rat) and not needed to answer structural questions, this notebook only opens each
`bvr` file, which already contains everything about trial timing and labels.

This is meant to run in well under a minute total.


In [4]:
import numpy as np
import pandas as pd
from pathlib import Path
import gc

raw_dir = Path('../data/raw')
session_dirs = sorted([p for p in raw_dir.iterdir() if p.is_dir()])
print(f"Found {len(session_dirs)} session folders:")
for p in session_dirs:
    print(" ", p.name)


Found 5 session folders:
  080718_mitt
  081106_barat
  090212_stella
  090212_superchris
  090420_buchanan


## Reference structure (from the Mitt audit)

We already confirmed Mitt's `bvr` file has 18 channels with these exact names, in this order. We'll
check every other rat against this list, if any rat's `bvr` file has different channels or a different
order, that's a signal we can't blindly reuse the same preprocessing code across all five rats.


In [5]:
EXPECTED_KEYS = [
    'TimeBin', 'Odor1', 'Odor2', 'Odor3', 'Odor4', 'Odor5',
    'Position1', 'Position2', 'Position3', 'Position4', 'Position5',
    'InSeqLog', 'PerformanceLog', 'PokeEvents', 'FrontReward', 'BackReward',
    'XvalRatMazePosition', 'YvalRatMazePosition',
]


## Run the audit loop across all rats

For each session: load `bvr`, confirm the channel names match, extract trial events from `InSeqLog`,
compute the InSeq/OutSeq and correct/incorrect splits, measure sampling rate and trial spacing, and check
odor-label cleanliness (every trial should have exactly one active odor channel, as we found for Mitt).

We delete each array and run garbage collection after processing so memory doesn't build up across all
five rats.


In [6]:
results = []

for session_dir in session_dirs:
    bvr_path = session_dir / f"{session_dir.name}_bvr.npz"
    if not bvr_path.exists():
        print(f"WARNING: no bvr file found for {session_dir.name}, skipping")
        continue

    bvr = np.load(bvr_path, allow_pickle=True)
    data = bvr['data']
    keys = bvr['keys'].tolist()

    keys_match = (keys == EXPECTED_KEYS)

    inseq_idx = keys.index('InSeqLog')
    perf_idx = keys.index('PerformanceLog')
    time_idx = keys.index('TimeBin')

    idx = np.where(data[inseq_idx] != 0)[0]
    inseq_vals = data[inseq_idx][idx]
    perf_vals = data[perf_idx][idx]

    n_inseq = int(np.sum(inseq_vals == 1))
    n_outseq = int(np.sum(inseq_vals == -1))
    n_correct = int(np.sum(perf_vals == 1))
    n_incorrect = int(np.sum(perf_vals == -1))

    timebin = data[time_idx]
    sampling_rate = 1 / np.diff(timebin).mean()
    duration_min = (timebin[-1] - timebin[0]) / 60

    gaps_sec = np.diff(idx) / sampling_rate
    min_gap = float(gaps_sec.min()) if len(gaps_sec) else None

    # odor label check: does every trial have exactly one active odor channel?
    odor_names = [k for k in keys if k.startswith('Odor')]
    odor_rows = [data[keys.index(name)] for name in odor_names]
    window_before = 200
    odor_counts = {name: 0 for name in odor_names}
    ambiguous = 0
    first_trial_odor = None
    for i, t in enumerate(idx):
        start = max(0, t - window_before)
        active = [j for j, row in enumerate(odor_rows) if row[start:t + 1].any()]
        if len(active) == 1:
            odor_counts[odor_names[active[0]]] += 1
            if i == 0:
                first_trial_odor = odor_names[active[0]]
        else:
            ambiguous += 1

    results.append({
        'session': session_dir.name,
        'keys_match_mitt': keys_match,
        'n_bvr_channels': len(keys),
        'n_trials': len(idx),
        'n_inseq': n_inseq,
        'n_outseq': n_outseq,
        'pct_outseq': round(100 * n_outseq / len(idx), 1) if len(idx) else None,
        'n_correct': n_correct,
        'n_incorrect': n_incorrect,
        'sampling_rate_hz': round(sampling_rate, 2),
        'duration_min': round(duration_min, 1),
        'min_trial_gap_sec': round(min_gap, 2) if min_gap else None,
        'ambiguous_odor_trials': ambiguous,
        'first_trial_odor': first_trial_odor,
    })

    print(f"done: {session_dir.name}")

    del bvr, data
    gc.collect()

df = pd.DataFrame(results)
df


done: 080718_mitt
done: 081106_barat
done: 090212_stella
done: 090212_superchris
done: 090420_buchanan


,session,keys_match_mitt,n_bvr_channels,n_trials,n_inseq,n_outseq,pct_outseq,n_correct,n_incorrect,sampling_rate_hz,duration_min,min_trial_gap_sec,ambiguous_odor_trials,first_trial_odor
0,080718_mitt,True,18,292,262,30,10.3,246,46,714.05,122.7,3.88,0,Odor1
1,081106_barat,True,18,176,155,21,11.9,165,11,769.54,81.3,3.00,0,Odor2
2,090212_stella,True,18,222,194,28,12.6,199,23,958.36,54.6,3.30,0,Odor1
3,090212_superchris,True,18,240,210,30,12.5,216,24,1000.00,51.1,2.15,0,Odor2
4,090420_buchanan,True,18,270,226,44,16.3,232,38,759.54,71.6,3.89,0,Odor3


## What to check in the output above

- **`keys_match_mitt`**: if this is `False` for any rat, stop and look, it means that rat's file has a
  different channel structure and needs individual handling before we can write one shared
  preprocessing function.
- **`ambiguous_odor_trials`**: should be 0 for every rat, same as Mitt. If not, that rat's odor labels
  aren't as clean and will need a wider search window or a different approach.
- **`pct_outseq`**: check whether the ~10% InSeq/OutSeq imbalance we saw in Mitt holds across rats, or
  whether some rats have a notably different balance, that affects whether pooling data across rats
  changes the class balance meaningfully.
- **`n_trials`**: tells us how much data each rat contributes, useful for deciding which rat to prioritize
  and for planning cross-validation folds.
- **`min_trial_gap_sec`**: confirm every rat has enough spacing for our planned window length, not just
  Mitt.
- **`first_trial_odor`**: a first pass at checking the "sequences always start with A" note from the
  meeting, this only checks the very first trial in the whole session though, not whether every
  *sequence* (a full ABCDE run) starts with A. Worth a closer look separately if this doesn't come back
  'Odor1' consistently, since a real check needs to group trials into sequences first, which is a bigger
  task for later, not part of this light pass.


## Summary

| session | keys match | trials | InSeq | OutSeq | % OutSeq | correct | incorrect | sampling rate (Hz) | duration (min) | min trial gap (s) | ambiguous odor trials |
|---|---|---|---|---|---|---|---|---|---|---|---|
| 080718_mitt | ✓ | 292 | 262 | 30 | 10.3% | 246 | 46 | 714.05 | 122.7 | 3.88 | 0 |
| 081106_barat | ✓ | 176 | 155 | 21 | 11.9% | 165 | 11 | 769.54 | 81.3 | 3.00 | 0 |
| 090212_stella | ✓ | 222 | 194 | 28 | 12.6% | 199 | 23 | 958.36 | 54.6 | 3.30 | 0 |
| 090212_superchris | ✓ | 240 | 210 | 30 | 12.5% | 216 | 24 | 1000.00 | 51.1 | 2.15 | 0 |
| 090420_buchanan | ✓ | 270 | 226 | 44 | 16.3% | 232 | 38 | 759.54 | 71.6 | 3.89 | 0 |



## Key notes from this audit


- **Sampling rate is NOT constant across rats** (ranges from 714 Hz to 1000 Hz). Thus, `segment_trials()` must always
  compute sampling rate fresh from each session's own `TimeBin` channel, never assume a shared constant.
- **Trial counts vary meaningfully by rat** (176 to 292), and each rat is its own recording session on
  its own day. This supports evaluating per-rat rather than blindly pooling all sessions into one
  dataset.
- **OutSeq trial percentage stays fairly consistent** across rats (10.3%–16.3%), confirming the class
  imbalance we saw in Mitt is a property of the task design, not specific to one rat.
- **Odor labels are clean for every rat** (0 ambiguous trials across all five), and channel structure
  (`keys_match_mitt`) is identical across all five `bvr` files, so one shared preprocessing pipeline is
  safe to build, as long as it respects the per-rat sampling rate difference above.